# 01 - Data acquisition, cleaning and quality checks

**Goal:** turn ~450 MB of raw Census LODES8 block-level files into clean, validated tract-level tables.

**Why tracts?** Raw files are keyed by 15-digit census *blocks* (~170k workplace blocks / 430k residence blocks in Texas). LEHD adds synthetic noise at block level to protect confidentiality, and blocks are too granular to interpret. Aggregating to 11-digit *tracts* (6,896 in Texas) removes ~96% of rows and damps the noise.

In [1]:
import sys, sqlite3
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.float_format', '{:,.2f}'.format)
from src.config import *


## 1. What a raw WAC file looks like

In [2]:
raw = pd.read_csv(RAW / 'wac' / 'tx_wac_S000_JT00_2023.csv.gz', dtype={'w_geocode': str})
print(raw.shape)
raw.iloc[:3, :12]

(168581, 53)


,w_geocode,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,CNS03,CNS04
0,480019501001005,23,3,11,9,3,8,12,0,0,0,0
1,480019501001009,119,18,61,40,5,35,79,0,0,0,0
2,480019501001010,37,12,14,11,12,22,3,0,0,0,0


`w_geocode` must be read as a **string** - blocks start with `48` (Texas) and identifiers lose meaning as integers. `C000` = total jobs; `CNS01-20` = NAICS supersectors; `CE01-03` = monthly-earnings bands; `CA01-03` = age bands.

## 2. Data-quality checks (run on every file)

`src/clean.py` asserts that identifiers are unique and well-formed, counts are non-negative and non-null, and that the sector, wage and age breakdowns each add back to the job total. Results for all 31 raw files:

In [3]:
qa = pd.read_json(INTERIM / 'qa.json')
checks = [c for c in qa.columns if c.endswith('mismatch') or c in ('dup_geocodes','dup_pairs','bad_geocode_len','negative_values','null_values','blocks_not_in_xwalk_tract','pairs_missing_coords')]
print('files checked:', len(qa), '| failed checks:', int(qa[checks].fillna(0).sum().sum()))
qa[['kind','year','rows_raw','total_jobs_raw','total_jobs_clean']].query('year in [2010, 2019, 2023]')

files checked: 31 | failed checks: 0


,kind,year,rows_raw,total_jobs_raw,total_jobs_clean
0,wac,2010,173822,10152390,10152390
9,wac,2019,174810,12433128,12433128
13,wac,2023,168581,13543073,13543073
14,rac,2010,425337,10130735,10130735
23,rac,2019,423474,12427692,12427692
27,rac,2023,431742,13543462,13543462
29,od,2019,11064354,12237260,3388835
30,od,2023,12039831,13335103,3682295


Zero failures. As a plausibility check, the 2019 Texas WAC total (12.4M) is in the range expected for jobs covered by unemployment insurance (roughly 12-13M).

## 3. Cleaning steps, demonstrated on one year

In [4]:
from src.clean import clean_lodes_area, build_dim_tract
dim, blocks = build_dim_tract()
wac23, qa23 = clean_lodes_area('wac', 2023, set(dim['tract']))
print(f"{qa23['rows_raw']:,} blocks -> {len(wac23):,} tracts | jobs preserved: {qa23['total_jobs_raw'] == wac23['jobs'].sum()}")
wac23.head()

168,581 blocks -> 6,880 tracts | jobs preserved: True


,year,tract,jobs,age_u30,age_30_54,age_55p,wage_low,wage_mid,wage_high,cns01,cns02,cns03,cns04,cns05,cns06,cns07,cns08,cns09,cns10,cns11,cns12,cns13,cns14,cns15,cns16,cns17,cns18,cns19,cns20
0,2023,48001950100,948,184,474,290,149,324,475,33,16,10,81,119,1,130,76,2,42,4,19,0,0,227,37,26,32,68,25
1,2023,48001950401,157,22,91,44,9,22,126,0,0,0,81,0,0,0,0,0,0,0,0,0,0,47,0,0,0,0,29
2,2023,48001950500,954,218,457,279,288,372,294,1,1,0,59,5,4,216,10,29,19,30,41,2,46,0,198,118,98,49,28
3,2023,48001950600,1194,280,613,301,275,402,517,2,73,6,58,43,53,200,263,0,16,50,4,0,49,0,70,0,274,33,0
4,2023,48001950700,1590,287,828,475,279,572,739,4,0,18,31,0,86,188,8,51,49,8,59,0,28,6,645,0,147,20,242


Total jobs are **exactly preserved** through aggregation (the check above prints `True`). Column names are standardised (`CE01` -> `wage_low`, `CNS16` -> `cns16`).

## 4. The tract dimension

Built from the LEHD geography crosswalk (names, county, CBSA) joined to TIGER polygons for true tract centroids and land area. The Houston MSA is flagged from its nine county FIPS codes.

In [5]:
dim[dim.in_houston_msa == 1].groupby('county_name').agg(tracts=('tract','count'), land_sqmi=('land_sqmi','sum')).sort_values('tracts', ascending=False)

,tracts,land_sqmi
county_name,,
Harris,1115,"1,707.02"
Fort Bend,133,861.77
Montgomery,133,"1,042.11"
Galveston,103,379.41
Brazoria,78,"1,363.27"
Liberty,17,"1,158.35"
Waller,11,513.30
Austin,8,646.50
Chambers,8,597.10


## 5. Sanity check: does the cleaned data tell a coherent story?

In [6]:
wac = pd.read_parquet(INTERIM / 'wac_tract.parquet').merge(dim[['tract','cbsa']], on='tract')
tx = wac.groupby('year')['jobs'].sum()
hou = wac[wac.cbsa == '26420'].groupby('year')['jobs'].sum()
pd.DataFrame({'texas': tx, 'houston_msa': hou, 'houston_share_%': 100 * hou / tx})

,texas,houston_msa,houston_share_%
year,,,
2010,10152390,2530059,24.92
2011,10476895,2636575,25.17
2012,10636736,2680966,25.20
2013,10957410,2780308,25.37
2014,11286517,2889420,25.60
2015,11575073,2986623,25.80
2016,11743249,2987075,25.44
2017,11925643,2990345,25.07
2018,12182193,3050225,25.04


Texas jobs dip in 2020 (COVID). Houston's share of the state rises from 24.9% to a 25.8% peak in 2015, then slides to 24.7% by 2023 - the oil-bust-then-lag pattern the SQL analysis quantifies. Next: model the tables in SQLite (`src/build_db.py`) and analyse them in SQL.